# ML pipeline for UNO

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision.io import decode_image
from torchvision.transforms import v2
from torch.utils.data import Dataset, DataLoader

from ML_utils import *
from image_utils import *

First let us define some useful global variables :
- File paths to the data
- Translations from text to vector representation
- ML hyperparameters

In [ ]:
# Filenames to access data
img_dir = os.path.join("augmented_train_set", "train_images")
annotations_file = os.path.join("augmented_train_set", "train") + '.csv'
checkpoint_dir = "checkpoints"
checkpoint_model_name = "model_10M_params"
if (not os.path.exists(checkpoint_dir)):
    os.mkdir(checkpoint_dir)
checkpoint_path = os.path.join(checkpoint_dir, checkpoint_model_name) + '.pt'
checkpoint_training_plot = os.path.join(checkpoint_dir, checkpoint_model_name) + '_plot.pdf'

# List of cards : digits (0 to 9) with color, specials (draw_2, reverse, skip) with color, other special (draw_4, wild)
card_name_list = [str(color) + '_' + str(digit) for digit in range(10) for color in ['r', 'g', 'b', 'y']]
card_name_list += [str(color) + '_' + str(special) for special in ['draw_2', 'reverse', 'skip'] for color in ['r', 'g', 'b', 'y']]
card_name_list += ['draw_4', 'wild']
# Mapping from card name to its index in vector form
card_name_to_index = {card_name : index for index, card_name in enumerate(card_name_list)}
# Define output size
label_output_size = len(card_name_list) + 1

# General ML parameters :
torch.manual_seed(0)
generator = torch.Generator().manual_seed(42)
train_val_test_split = [0.7, 0.15, 0.15]
learning_rate = 0.01
batch_size = 128 # True batch size (number of samples per gradient descent updates)
batch_size_simul = 8 # Number of samples loaded simultaneously (to minimize memory consumption)
nb_epochs = 10
p_dropout = 0.0

RGB_normalization_mean = torch.tensor([0.485, 0.456, 0.406])
RGB_normalization_std = torch.tensor([0.229, 0.224, 0.225])
transforms = v2.Compose([
    v2.RandomApply(transforms=[v2.RandomRotation((180, 180))], p=0.5),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=RGB_normalization_mean, std=RGB_normalization_std),
])

In [ ]:
def image_reverse_RGB_normalization(img):
    RGB_std = RGB_normalization_std.reshape((3,1,1)).to(img.device)
    RGB_mean = RGB_normalization_mean.reshape((3,1,1)).to(img.device)
    return (255*(img * RGB_std + RGB_mean)).to(torch.uint8)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device {device}")

Then let us define helper functions to handle transformations between data and ML model.

In [ ]:
def output_transform(model_output):
    return model_output.softmax(dim=1)

def label_string_to_tensor(label):
    # Give the model representation from the label annotation :
    output = torch.zeros(label_output_size)
    if (label == 'EMPTY'):
        output[-1] = 1
    else:
        for card_name in label.split(";"):
            output[card_name_to_index[card_name]] += 1
    output = output / torch.sum(output)
    return output

def label_tensor_to_string(vector):
    # Give the label annotation from the model representation :
    card_name_list_completed = card_name_list + ['EMPTY']
    card_list = []
    # Go from probabilities to number of cards
    vector_list = list((vector / torch.max(vector)).flatten())
    for index, value in enumerate(vector_list):
        nb_cards = round(value.item())
        while (nb_cards > 0):
            card_list.append(card_name_list_completed[index])
            nb_cards -= 1
    output = ';'.join(card_list)
    return output

In [ ]:
# Take a look a the image in the dataset
full_image_example = decode_image(get_random_imagepath(img_dir))
print(f"Image shape {full_image_example.shape}")
divided_images_example = [divide_image(full_image_example, i) for i in range(5)]
print(f"Divided image shape {divided_images_example[0].shape}")

display_tensor_images(full_image_example)
display_tensor_images(divided_images_example, transform=None, shape=(1, len(divided_images_example)), figsize=(5,10))

In [ ]:
class UNOImageDataset(Dataset):
    def __init__(self, img_dir, annotations_file, divide_image, label_string_to_tensor, transform=None, device='cpu'):
        # Annotations contains : 'image_id', 'center_card', 'active_player', 'player_1_cards', 'player_2_cards', 'player_3_cards', 'player_4_cards'
        annotations = pd.read_csv(annotations_file)
        #annotations = annotations[:81] # Only original dataset
        self.device = device
        self.transform = transform
        # Store image labels
        self.img_labels = torch.zeros((5*len(annotations['image_id']), label_output_size), dtype=torch.float32)
        self.img_labels[0::5] = torch.tensor([list(label_string_to_tensor(label)) for label in annotations['center_card']])
        self.img_labels[1::5] = torch.tensor([list(label_string_to_tensor(label)) for label in annotations['player_1_cards']])
        self.img_labels[2::5] = torch.tensor([list(label_string_to_tensor(label)) for label in annotations['player_2_cards']])
        self.img_labels[3::5] = torch.tensor([list(label_string_to_tensor(label)) for label in annotations['player_3_cards']])
        self.img_labels[4::5] = torch.tensor([list(label_string_to_tensor(label)) for label in annotations['player_4_cards']])
        # Store images
        self.imgs = torch.zeros((5*len(annotations['image_id']), 3, 886, 2228), dtype=torch.uint8)
        for img_index in range(len(annotations['image_id'])):
            img_path = os.path.join(img_dir, annotations['image_id'][img_index]) + '.jpg'
            full_image = decode_image(img_path)
            for divided_img_index in range(5):
                self.imgs[img_index*5 + divided_img_index] = divide_image(full_image, divided_img_index)

    def __len__(self):
        return self.img_labels.shape[0]

    def __getitem__(self, idx):
        image = self.imgs[idx]
        if (self.transform != None):
            image = self.transform(image)
        image = image.to(self.device)
        label = self.img_labels[idx].to(self.device)
        return image, label

In [ ]:
full_dataset = UNOImageDataset(img_dir, annotations_file, divide_image, label_string_to_tensor, transform=transforms, device=device)
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(full_dataset, train_val_test_split, generator=generator)

# Display sample
image_sample, label_sample = full_dataset.__getitem__(2)
with torch.no_grad():
    display_tensor_images(image_sample, transform=image_reverse_RGB_normalization)
    print(label_sample)
    print(label_tensor_to_string(label_sample))
print(train_dataset.__len__())

## Define the model

In [ ]:
def conv2d_resblock(nb_channels, kernel_size, padding_mode='reflect'):
    padding = kernel_size//2
    layers_list = [nn.Conv2d(in_channels=nb_channels, out_channels=nb_channels, kernel_size=kernel_size, 
                             padding=kernel_size//2, padding_mode=padding_mode), 
                   nn.BatchNorm2d(nb_channels), 
                   nn.ReLU(inplace=True), 
                   nn.Conv2d(in_channels=nb_channels, out_channels=nb_channels, kernel_size=kernel_size, 
                             padding=kernel_size//2, padding_mode=padding_mode), 
                   nn.BatchNorm2d(nb_channels), 
                   nn.ReLU(inplace=True)]
    return nn.Sequential(*layers_list)

class UNO_Model(nn.Module):
    def __init__(self):
        super().__init__()
        # Recall shape 3, 886, 2228
        self.conv1 = nn.Sequential(nn.Conv2d(3, 64, 7), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.conv2 = nn.Sequential(nn.Conv2d(64, 64, 3), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.conv3 = nn.Sequential(nn.Conv2d(64, 128, 3), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.conv4 = nn.Sequential(nn.Conv2d(128, 256, 3), nn.BatchNorm2d(256), nn.ReLU(inplace=True))
        self.conv5 = nn.Sequential(nn.Conv2d(256, 512, 3), nn.BatchNorm2d(512), nn.ReLU(inplace=True))
        self.conv6 = nn.Sequential(nn.Conv2d(512, 1024, 3), nn.BatchNorm2d(1024), nn.ReLU(inplace=True))

        self.initPooling = nn.AvgPool2d((5,5))
        self.maxPooling = nn.MaxPool2d((2,2))
        self.avgPooling = nn.AdaptiveAvgPool2d((1, 1))
        
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256), 
            nn.ReLU(inplace=True), 
            nn.Linear(256, 128), 
            nn.ReLU(inplace=True), 
            nn.Linear(128, label_output_size), 
        )

    def forward(self, x):
        x = self.initPooling(x)
        #print(x.shape)
        #display_tensor_images(image_reverse_RGB_normalization(x[0]))
        x = self.conv1(x)
        x = self.maxPooling(x)
        x = self.conv2(x)
        x = self.maxPooling(x)
        x = self.conv3(x)
        x = self.maxPooling(x)
        x = self.conv4(x)
        x = self.maxPooling(x)
        x = self.conv5(x)
        x = self.maxPooling(x)
        x = self.conv6(x)
        #print(x.shape)
        x = self.avgPooling(x)
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = self.classifier(x)
        return x

model = UNO_Model()
torch.manual_seed(0)
model.apply(init_weights)
model = model.to(device)
print(model)
print(f"Number of parameters : {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

In [ ]:
with torch.no_grad():
    print(output_transform(model(image_sample.unsqueeze(0))))
    print(label_sample)

## Train the model

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, nb_epochs)

In [ ]:
train_output = train(model, train_dataset, val_dataset, batch_size, nb_epochs, optimizer, scheduler, loss_fn, 
                     output_transform, checkpoint_path, show_plot=True, batch_size_simul=batch_size_simul, generator=generator)
best_model, best_f1, best_epoch, val_f1s, val_losses, train_losses = train_output
print(f"Best model at epoch {best_epoch} -> {100*best_f1:.2f}% F1 score")

In [ ]:
plot_training(best_epoch, val_f1s, val_losses, train_losses, first_epoch_plot=1, save_filename=checkpoint_training_plot)

In [ ]:
with torch.no_grad():
    model = UNO_Model().to(device)
    checkpoint = torch.load(checkpoint_path)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    label_predict = output_transform(model(image_sample.unsqueeze(0)))
    print(label_predict)
    print(label_tensor_to_string(label_predict))
    print(label_sample)
    print(label_tensor_to_string(label_sample))